# L66 · ASR 系统全览 — 声学模型（acoustic model，AM） → 语言模型（language model，LM） → 解码器（decoder），端到端（end-to-end，E2E） vs 经典流水线（pipeline）

**目标**：建立现代端到端 ASR 的完整系统观——特征提取（feature extraction） → 编码器（encoder） → 解码器 → 文字，理解各环节的输入/输出形状与作用。

🔗 **Aurora 连接**：`aurora.audio.mel` 实现了 ASR 的特征提取层（log-Mel 频谱（log-Mel spectrogram））；`aurora.speech`（本月新建）将承载 CTC/Attention 编解码器。

← **上一课**　[L65 · 训练可视化](../6_deep_learning/L65_visual_training.ipynb)

> 上节课学习了 **训练可视化**：Loss / Acc 曲线，梯度范数，权重分布直方图。  
> 本课将探讨 **ASR 系统全览**。

现代 ASR 把一段连续波形变成离散词序列。端到端模型（Whisper、wav2vec 2.0）把原本分开优化的声学模型、语言模型合并成一个大网络，唯一的监督信号是配对的（音频, 文字）样本。理解三个支柱——**特征**、**序列建模**、**评估指标**——是本月的主线。

In [ ]:
import numpy as np

## 1. 端到端 ASR vs 传统 HMM-GMM

传统流水线把任务切成三段独立训练：**声学模型**（GMM 建模帧概率）→ **发音词典（pronunciation dictionary / lexicon）** → **语言模型**（N-gram）。误差无法端到端反传，各段最优不代表整体最优。

端到端系统（Whisper/wav2vec 2.0）直接学 P(文字 | 音频)：Transformer 编码器处理帧序列，解码器（或 CTC 头）输出 token 概率。一次梯度下降（gradient descent）同时优化全链路，数据量够大时效果碾压传统方案。

架构对比（简化）：
```
HMM-GMM: waveform → MFCC → GMM/HMM → 词典 → N-gram LM → text
Whisper:  waveform → log-Mel → Transformer Enc → Transformer Dec → text
```

In [ ]:
# 演示：两种路径的「接口」差异
# 传统方案输出每帧的 HMM 状态后验概率；端到端直接输出 token logits

T, n_mels = 300, 80   # 300 帧，80维 log-Mel
n_vocab   = 51865     # Whisper 词表大小

# 传统：每帧 → GMM 状态概率（假设 3000 个 HMM 状态）
n_hmm_states = 3000
hmm_output_shape = (T, n_hmm_states)

# 端到端：编码器输出序列 → 解码器输出 token 概率
enc_output_shape = (T // 2, 512)   # 2× 下采样后，d_model=512
dec_output_shape = (n_vocab,)       # 每步预测一个 token

print(f'HMM-GMM 每帧输出维度: {hmm_output_shape}')
print(f'Whisper 编码器输出形状: {enc_output_shape}')
print(f'Whisper 解码器每步输出维度: {dec_output_shape}')

### 1b. 三大模块的输入/输出形状与参数量

端到端 ASR 可以拆解为三个可复用模块，理解各模块的接口是系统集成的基础：

| 模块 | 输入 | 输出 | 典型参数量（Whisper-small） |
|---|---|---|---|
| 声学编码器（AM/Encoder） |  log-Mel 频谱 |  隐状态序列 | ≈ 77 M |
| 语言解码器（LM/Decoder） | 编码器输出 + 已生成 token |  下一词 logits | ≈ 84 M |
| 束搜索解码器（Beam Decoder） | logits 序列 | token id 列表 → 文字 | 无可训练参数 |

传统 HMM-GMM 中，三段**分开训练**，误差无法端到端反传；
Whisper 等 E2E 模型用统一的交叉熵损失对齐三段，显著降低工程复杂度。

In [ ]:
# 演示：AM / LM / Decoder 的尺寸变换
B, T, n_mels = 2, 300, 80
encoder_d = 512
vocab_size = 51865

# Step 1 — 声学编码器：T 帧 → T/2 帧（Whisper 用 2 个 stride-2 conv 下采样）
am_out_shape = (B, T // 2, encoder_d)
print(f"AM  输出: {am_out_shape}")

# Step 2 — 语言解码器：每步生成 1 个 token（自回归）
lm_out_shape = (B, vocab_size)
print(f"LM  输出（每步）: {lm_out_shape}")

# Step 3 — 束搜索：保留 beam_size 条候选路径
beam_size = 5
beam_candidates = beam_size
print(f"束搜索: 每步保留 {beam_candidates} 条候选")

# AM / LM 参数量粗估（Whisper-small）
am_params  = 77e6
lm_params  = 84e6
print(f"Whisper-small: AM ≈ {am_params/1e6:.0f}M 参数，LM ≈ {lm_params/1e6:.0f}M 参数")
print(f"总参数量 ≈ {(am_params + lm_params)/1e6:.0f}M")


## 2. 特征：80-dim log-Mel 频谱

Whisper 的标准输入是 **80维 log-Mel 频谱**，帧参数与 L43-L47 完全一致：
```
采样率   sr = 16000 Hz
窗长     n_fft  = 400  样本  （25 ms）
帧移     hop    = 160  样本  （10 ms）
Mel bins n_mels = 80
```

处理流程：`stft → |·|² → mel_filterbank · power → log(max(·, 1e-10))`。结果形状为 `(80, T_frames)`，T_frames ≈ 总样本数 / hop。

为什么 Mel 而非线性频率？人耳对低频分辨率更高，Mel 尺度模拟这一非线性，让模型更容易学到音素（phoneme）边界。

In [ ]:
# 演示：计算一段随机信号的 log-Mel 频谱形状
sr     = 16000
n_fft  = 400
hop    = 160
n_mels = 80
duration_s = 3.0

n_samples  = int(sr * duration_s)
n_frames   = 1 + (n_samples - n_fft) // hop

print(f'音频长度: {duration_s} s  →  {n_samples} 样本')
print(f'STFT 帧数 (n_fft=400, hop=160): {n_frames}')
print(f'log-Mel 频谱形状: ({n_mels}, {n_frames})')
print(f'每帧覆盖: {n_fft/sr*1000:.1f} ms，帧移: {hop/sr*1000:.1f} ms')

## 3. WER — 词错率

WER（Word Error Rate）是 ASR 最核心的评估指标：
```
WER = (S + D + I) / N
```
- `N`：参考（正确）文本的总词数
- `S`（Substitution）：替换错误数——模型输出了错词
- `D`（Deletion）：删除错误数——漏说了某词
- `I`（Insertion）：插入错误数——多说了某词

WER 通过**编辑距离**（Levenshtein distance）计算，以词为单位而非字符。WER 可以 > 1.0（大量插入时）。

常见基线（英语）：
```
Whisper-large-v3    WER ≈ 2–5%   (LibriSpeech test-clean)
Whisper-base        WER ≈ 5–8%
传统 HMM-GMM        WER ≈ 8–15%
```

In [ ]:
# 演示：手工追踪一个 WER 计算例子
reference  = ['the', 'cat', 'sat', 'on', 'the', 'mat']
hypothesis = ['the', 'cat', 'sat', 'on', 'a',   'mat']
#                                           ^ 替换: the→a

# 手工统计
S = 1   # 'the' → 'a'
D = 0
I = 0
N = len(reference)
wer_manual = (S + D + I) / N

print(f'参考: {reference}')
print(f'假设: {hypothesis}')
print(f'S={S}, D={D}, I={I}, N={N}')
print(f'WER = ({S}+{D}+{I})/{N} = {wer_manual:.4f} = {wer_manual*100:.2f}%')

## 4. ✏️ 实现 `compute_wer(hypothesis, reference)`

**推理路线**：
1. 用动态规划（dynamic programming，DP）构建编辑距离矩阵：`dp[i][j]` 表示把 `hyp[:i]` 变成 `ref[:j]` 所需的最少操作数（替换/删除/插入各计 1 次）
2. 状态转移：若 `hyp[i-1] == ref[j-1]` 则 `dp[i][j] = dp[i-1][j-1]`；否则 `dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])`
3. 总错误数 = `dp[len(hyp)][len(ref)]`，除以 `len(reference)` 得到 WER

**参考输入输出**：
```
hyp = ['the', 'cat', 'sat']
ref = ['the', 'sat', 'cat']
编辑距离 = 2（cat→sat 顺序互换：Levenshtein 视为 2 次替换，无 swap 操作）
WER = 2 / 3 ≈ 0.6667
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


> **💡 学习顺序说明**：本练习采用先试后学策略——先尝试实现 DP，建立直觉；
> L67 将系统推导 Levenshtein 算法的数学基础。
> 如果现在卡住，可跳到 L67 学完再回来完成此实现，两种顺序均可。
>
> 边界条件规范（实现时必须处理）：
> -  为空且  亦为空 → 返回 
> -  为空但  非空 → 返回 （无法用 0 个词作参考时错率无穷）

In [ ]:
def compute_wer(hypothesis: list, reference: list) -> float:
    """计算词错率 WER = (S+D+I) / len(reference)。

    hypothesis: 模型输出的词列表
    reference:  正确文本的词列表
    """
    H, R = len(hypothesis), len(reference)
    # ✏️ TODO: 边界条件（R==0 时返回 0.0 或 float('inf')）

    # ✏️ TODO: 初始化 dp 数组（长度 R+1，表示把空序列变成 ref[:j] 的代价）

    # ✏️ TODO: 遍历 hypothesis 的每个词，更新 dp（滚动数组，O(R) 空间）

    # ✏️ TODO: return dp[R] / R
    raise NotImplementedError(
        "请实现 compute_wer：参考上方推导补全所有 TODO 后再运行此单元。"
        "如需提示，可先完成 L67 再回来实现。"
    )

In [ ]:
try:
    # 检查 compute_wer
    assert abs(compute_wer(['the', 'cat'], ['the', 'cat']) - 0.0) < 0.001, '完全一致 WER 应为 0'
    assert abs(compute_wer([], ['a']) - 1.0) < 0.001, '全删除 WER 应为 1'
    assert abs(compute_wer(['the', 'cat', 'sat'], ['the', 'sat', 'cat']) - 2/3) < 0.001, \
        '顺序互换（2次替换）WER 应为 2/3'
    assert abs(compute_wer(['a', 'b', 'c', 'd'], ['b', 'c']) - 1.0) < 0.001, \
        '多余词导致插入错误'
    print('✅ compute_wer 全部检查通过')
except NotImplementedError as _e:
    print(f"⚠️  compute_wer 尚未实现 — 完成 TODO 后重新运行：{_e}")

## 5. 参数实验：错误类型分析

模拟 10 句话的 ASR 转写结果，统计 S/D/I 分布，观察不同类型错误对 WER 的贡献。

**预期现象**：
- 背景噪声场景：**D（删除）** 占比高，弱读音节被吞掉
- 口音场景：**S（替换）** 占比高，音素混淆
- 语速极快场景：**D** 再次升高，词边界丢失

实验中修改 `pairs` 列表，替换成自己的转写结果，观察统计变化。

In [ ]:
# 模拟 10 句话的参考文本与 ASR 假设（手工输入或真实转写）
pairs = [
    # (hypothesis_words, reference_words)
    (['hello', 'world'],                    ['hello', 'world']),           # 完美
    (['the', 'cat', 'sat'],                 ['the', 'cat', 'sat', 'down']),# 删除
    (['i', 'love', 'music'],                ['i', 'love', 'music']),       # 完美
    (['this', 'is', 'a', 'test'],           ['this', 'is', 'test']),       # 插入
    (['speech', 'recognition'],             ['speech', 'recognition']),    # 完美
    (['the', 'whether', 'is', 'nice'],      ['the', 'weather', 'is', 'nice']),# 替换
    (['aurora', 'audio'],                   ['aurora', 'audio', 'ai']),    # 删除
    (['deep', 'learning', 'rocks'],         ['deep', 'learning', 'rocks']),# 完美
    (['transformer', 'model'],              ['transformer', 'models']),    # 替换
    (['end', 'to', 'end', 'system'],        ['end', 'to', 'end', 'system']),# 完美
]

wers = [compute_wer(h, r) for h, r in pairs]
avg_wer = np.mean(wers)

print('句子 WER 分布：')
for i, (w, (h, r)) in enumerate(zip(wers, pairs), 1):
    status = '✅' if w == 0 else '❌'
    print(f'  句{i:2d} {status}  WER={w:.3f}  ref_len={len(r)}')
print(f'\n平均 WER = {avg_wer:.4f} = {avg_wer*100:.2f}%')

## 本课收束

本节实现了 `compute_wer`，它通过 Levenshtein 动态规划计算词级编辑距离，输出 `(S+D+I)/N` 浮点 WER 值。该指标将贯穿 `aurora.speech` 模块的全部训练与评测循环。下一课：**L67** 将从零实现 Levenshtein 编辑距离动态规划算法，建立 WER 的数学基础。

---

→ **下一课**　[L67 · Edit Distance 从零实现](L67_edit_distance.ipynb)

> 下节课将学习 **Edit Distance 从零实现**：Levenshtein 动态规划，WER 的数学基础。